# 01 AMD ROCm 设备与环境确认

        这一节的目标是先证明设备、环境、缓存和权限链路是可用的。很多训练失败看起来像模型问题，实际可能是 ROCm 没识别 GPU、缓存放错磁盘、网络无法下载权重，或者统一内存被其它进程挤占。

        大家在 Notebook 里主要做三件事：检查硬件和 PyTorch、规划大文件目录、形成一张设备资源表。


In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
NOTEBOOK_DIR = TOPIC_ROOT / "notebooks"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/every-embodied/mujoco_pnp"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / "outputs"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", PROJECT_ROOT / "ckpt"))

print("TOPIC_ROOT =", TOPIC_ROOT)
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT =", DATA_ROOT)
print("OUTPUT_ROOT =", OUTPUT_ROOT)
print("MODEL_ROOT =", MODEL_ROOT)


In [ ]:
try:
    from IPython.display import Image, Markdown, display
except Exception:
    class Markdown(str):
        pass

    def display(obj):
        print(obj)

    def Image(filename=None, width=None):
        return f"[image] {filename}"


def show_asset(filename, width=960):
    path = ASSET_DIR / filename
    if path.exists():
        display(Image(filename=str(path), width=width))
    else:
        print(f"缺少素材：{path}")


def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


## Checkpoint 1：ROCm 和 PyTorch 是否能看到 GPU


In [ ]:
def run_cmd(cmd):
    print("$", " ".join(cmd))
    if shutil.which(cmd[0]) is None:
        print(f"未找到命令：{cmd[0]}")
        return
    result = subprocess.run(cmd, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(result.stdout)


run_cmd(["rocm-smi", "--showuse", "--showtemp", "--showmemuse"])

try:
    import torch
    print("torch =", torch.__version__)
    print("torch.cuda.is_available() =", torch.cuda.is_available())
    print("torch.cuda.device_count() =", torch.cuda.device_count())
    if torch.cuda.is_available():
        print("device =", torch.cuda.get_device_name(0))
except Exception as exc:
    print("PyTorch 检查失败：", repr(exc))


如果 `rocm-smi` 正常、`torch.cuda.is_available()` 为 `True`，说明 ROCm 与 PyTorch 的基础链路已经打通。若前者正常但后者为 `False`，优先检查 PyTorch 是否为 ROCm build。


## Checkpoint 2：统一内存、显存和磁盘目录


In [ ]:
paths = [
    ("项目源码", PROJECT_ROOT),
    ("数据目录", DATA_ROOT),
    ("模型与 checkpoint", MODEL_ROOT),
    ("Notebook 输出", OUTPUT_ROOT),
    ("Hugging Face cache", Path(os.environ.get("HF_HOME", "/path/to/cache/huggingface"))),
]
md_table(["项目", "建议路径", "是否存在"], [(name, f"`{path}`", path.exists()) for name, path in paths])


大模型缓存、数据集、checkpoint 和批量视频都可能很大。大家只需要在报告里说明目录规划和磁盘类型，不需要把这些文件放进教程目录。


## Checkpoint 3：设备资源表模板


In [ ]:
rows = [
    ("设备型号", "AMD Ryzen AI MAX+ / Radeon GPU"),
    ("系统版本", "Ubuntu / WSL / 其它"),
    ("ROCm 版本", "填写 rocm-smi 或包版本"),
    ("PyTorch 版本", "填写 torch.__version__"),
    ("空闲温度", "例如 30-45 C"),
    ("训练温度", "例如 75-85 C"),
    ("ACT 训练 VRAM", "填写 rocm-smi 观察值"),
    ("SmolVLA 训练 VRAM", "填写 rocm-smi 观察值"),
    ("pi0 smoke 状态", "未跑 / 通过 / 失败原因"),
]
md_table(["项目", "记录"], rows)


完成本节后，大家应当能判断这台 AMD 设备是否已经具备继续训练和评估的条件。
